# Logistic Regression

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import os
import pickle

from sklearn.model_selection import cross_val_score, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score, 
                             f1_score, roc_auc_score)

import warnings
warnings.filterwarnings('ignore')

## Load Data

In [ ]:
# 1. CẤU HÌNH ĐƯỜNG DẪN
data_dir = '../data/'
target_col = 'class'  # Tên cột nhãn (0: thật, 1: giả)

# 2. LOAD DỮ LIỆU ĐÃ SCALED
train_df = pd.read_csv(os.path.join(data_dir, 'train_data.csv'))
test_df = pd.read_csv(os.path.join(data_dir, 'test_data.csv'))

X_train = train_df.drop(columns=[target_col])
y_train = train_df[target_col]

X_test = test_df.drop(columns=[target_col])
y_test = test_df[target_col]

print(f"Training data shape: {X_train.shape}")
print(f"Testing data shape: {X_test.shape}")

###  Danh sách lưu kết quả để xuất CSV

In [ ]:
results_list = []

def evaluate_model(model, name, X_test, y_test):
    """Dự đoán và trích xuất các chỉ số phân loại theo yêu cầu (Acc, Prec, Rec, F1, AUC)"""
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1] # Lấy xác suất của class 1 để tính AUC
    
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_prob)
    
    res = {
        'Algorithm': name,
        'Accuracy': round(acc, 2),
        'Precision': round(prec, 2),
        'Recall': round(rec, 2),
        'F1-Score': round(f1, 2),
        'AUC': round(auc, 2)
    }
    
    results_list.append(res)
    print(f"[{name}] Acc: {acc:.2f} | Prec: {prec:.2f} | Rec: {rec:.2f} | F1: {f1:.2f} | AUC: {auc:.2f}")
    return res


### 3. CHẠY BASELINE MODEL (Tham số mặc định)

In [ ]:
print("--- Training Baseline Logistic Regression ---")
lr_baseline = LogisticRegression(random_state=42)

# 5-Fold Cross Validation trên tập Train
cv_scores = cross_val_score(lr_baseline, X_train, y_train, cv=5, scoring='accuracy')
print(f"5-Fold CV Accuracy (Train): {np.mean(cv_scores):.4f} (+/- {np.std(cv_scores):.4f})")

# Huấn luyện và Đánh giá trên tập Test
lr_baseline.fit(X_train, y_train)
evaluate_model(lr_baseline, "Logistic Regression (Baseline)", X_test, y_test)

--- Đang huấn luyện Baseline ---
===> LR_Baseline - Accuracy: 0.9926 | TP: 123, FP: 2, TN: 145, FN: 0


### 4. CHẠY TUNING MODEL (Dùng GridSearchCV)

In [ ]:
print("--- Tuning Logistic Regression ---")

param_grid = {
    'C': [0.001, 0.01, 0.1, 1, 10, 100],
    'penalty': ['l1', 'l2'],
    'solver': ['liblinear', 'saga'] 
}

grid_search = GridSearchCV(
    LogisticRegression(random_state=42, max_iter=1000), 
    param_grid, 
    cv=5, 
    scoring='accuracy', 
    n_jobs=-1
)

grid_search.fit(X_train, y_train)
lr_tuned = grid_search.best_estimator_

print(f"Best Parameters: {grid_search.best_params_}")
print(f"Best 5-Fold CV Accuracy: {grid_search.best_score_:.4f}")

# Đánh giá model đã tune trên tập Test
evaluate_model(lr_tuned, "Logistic Regression (Tuned)", X_test, y_test)

V1 Accuracy (Test): 0.99
V2 Best Params: {'C': 100, 'penalty': 'l2'}


### (5) Lưu kết quả

In [ ]:
# CẤU HÌNH ĐƯỜNG DẪN LƯU
save_path = '../experiment/LR/'
os.makedirs(save_path, exist_ok=True)

# 1. TỔNG HỢP & LƯU CSV
df_results = pd.DataFrame(results_list)
csv_output = os.path.join(save_path, 'lr_evaluation_results.csv')
df_results.to_csv(csv_output, index=False)

# 2. LƯU MODELS (.pkl)
with open(os.path.join(save_path, 'lr_baseline.pkl'), 'wb') as f:
    pickle.dump(lr_baseline, f)

with open(os.path.join(save_path, 'lr_tuned.pkl'), 'wb') as f:
    pickle.dump(lr_tuned, f)

print("-" * 40)
print(f"✅ Đã lưu file CSV tại: {csv_output}")
print(f"✅ Đã lưu models tại: {save_path}")
print("-" * 40)

display(df_results)

TypeError: got an unexpected keyword argument 'squared'

In [ ]:
!jupyter nbconvert --to html LR_model.ipynb

[NbConvertApp] Converting notebook LR_model.ipynb to html
[NbConvertApp] Writing 299772 bytes to LR_model.html
